# 📊 Student Performance Analysis

A small, self-contained data-analysis notebook for testing **NotebookMind**.
It builds a synthetic dataset of students and explores how study and sleep relate to exam scores.

All randomness lives in one seeded cell, so every run is reproducible.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("Setup complete — numpy", np.__version__, "| pandas", pd.__version__)

In [ ]:
# Build a reproducible synthetic dataset of 40 students.
# Every random draw happens here, right after the seed, so this cell is idempotent.
np.random.seed(7)
n = 40

students = pd.DataFrame({
    "student_id": range(1, n + 1),
    "study_hours": np.random.randint(0, 8, size=n),
    "sleep_hours": np.random.normal(7, 1.2, size=n).round(1),
    "group": np.random.choice(["A", "B", "C"], size=n),
    "noise": np.random.normal(0, 5, size=n),
})

print(students.head())

In [ ]:
# Derive the exam score from study + sleep plus the pre-drawn noise.
# Pure transform of existing columns -> running it again gives the same result.
students["exam_score"] = (
    40
    + 5.0 * students["study_hours"]
    + 2.0 * students["sleep_hours"]
    + students["noise"]
).clip(0, 100).round(1)

print(students[["study_hours", "sleep_hours", "exam_score"]].describe())

In [ ]:
# Flag who passed (score >= 60) and report the pass rate.
students["passed"] = students["exam_score"] >= 60
pass_rate = students["passed"].mean()

print(f"Pass rate: {pass_rate:.1%}")
print(students["passed"].value_counts())

In [ ]:
# Compare the three study groups.
group_summary = students.groupby("group").agg(
    avg_score=("exam_score", "mean"),
    avg_study=("study_hours", "mean"),
    n_students=("student_id", "count"),
).round(2)

print(group_summary)

In [ ]:
# Which feature is most correlated with the exam score?
corr = students[["study_hours", "sleep_hours", "exam_score"]].corr()
print(corr.round(2))

strongest = corr["exam_score"].drop("exam_score").idxmax()
print("Most correlated with exam_score:", strongest)

In [ ]:
# Visualise the relationship between study hours and exam score.
fig, ax = plt.subplots(figsize=(7, 4))
colors = students["passed"].map({True: "#266DF0", False: "#F04438"})
ax.scatter(students["study_hours"], students["exam_score"], c=colors, alpha=0.8)
ax.set_xlabel("Study hours")
ax.set_ylabel("Exam score")
ax.set_title("Study hours vs. exam score")
plt.show()

In [ ]:
# Report the single best-performing student.
top = students.sort_values("exam_score", ascending=False).iloc[0]

print("Top student:")
print(f"  id={int(top['student_id'])}, group={top['group']}, score={top['exam_score']}")